# Machine Learning for CSI 300 Index Futures: Direction, Volatility, and Sentiment Prediction

**Abstract:** This project applies a comprehensive suite of machine learning and deep learning algorithms to predict three distinct aspects of China's CSI 300 Index Futures: price direction (classification), realized volatility (regression), and sentiment breakthrough events (classification). We benchmark 10 models per task — spanning linear models, tree-based ensembles, and neural architectures — and evaluate trading strategies derived from model predictions.

- **Training Period:** 2015–2023
- **Testing Period:** 2024–2025 (True Out-of-Sample)
- **Models:** Decision Tree, Logistic/Linear Regression, SVM, Random Forest, Gradient Boosting, LightGBM, XGBoost, CatBoost, LSTM variants, 1D-CNN
- **Evaluation:** Accuracy, AUC, F1, R², RMSE, MAE, Sharpe Ratio, Maximum Drawdown

## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Task 1 — Price Direction Prediction (Classification)](#2-task-1--price-direction-prediction-classification)
3. [Task 2 — Volatility Prediction (Regression)](#3-task-2--volatility-prediction-regression)
4. [Task 3 — Sentiment Breakthrough Prediction (Classification + Trading)](#4-task-3--sentiment-breakthrough-prediction-classification--trading)
5. [Summary and Conclusions](#5-summary-and-conclusions)

---
## 1. Environment Setup

Install dependencies and configure the runtime environment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

# ML frameworks
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    RandomForestRegressor, GradientBoostingRegressor
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
    mean_squared_error, r2_score, mean_absolute_error
)
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Interpretability
import shap

warnings.filterwarnings('ignore')

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({
    'figure.figsize': (14, 8),
    'font.size': 11,
    'font.family': 'DejaVu Sans',
    'axes.unicode_minus': False
})

OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Environment ready.')
print(f'TensorFlow version: {tf.__version__}')
print(f'Output directory: {OUTPUT_DIR}')

---
## 2. Task 1 — Price Direction Prediction (Classification)

**Objective:** Predict whether the CSI 300 futures closing price will increase or decrease on the next trading day.

**Target variable:** `direction = 1` if next-day return > 0, else `0`.

**Models evaluated:** Decision Tree, Gradient Boosting, SVM (Linear), Logistic Regression, SVM (RBF), LightGBM, Simple LSTM, Bidirectional LSTM, LSTM + Attention, 1D-CNN.

### 2.1 Data Loading and Target Generation

In [ ]:
# ============================================================================
# Data Loading — Direction Prediction
# ============================================================================
# Update these paths to your local data files
TRAIN_FILE = 'data/train_2015_2023.xlsx'
TEST_FILE  = 'data/test_2024_2025.xlsx'

train = pd.read_excel(TRAIN_FILE, sheet_name='Sheet1')
test  = pd.read_excel(TEST_FILE,  sheet_name='Sheet1')

print(f'Training set shape: {train.shape}')
print(f'Testing set shape:  {test.shape}')

# Generate target variable: next-day price direction (1 = up, 0 = down)
train['direction'] = (train['close_price'].pct_change().shift(-1) > 0).astype(int)
test['direction']  = (test['close_price'].pct_change().shift(-1) > 0).astype(int)

# Define feature columns (exclude identifiers, prices, and target)
cols_to_remove = [
    'code', 'name', 'date', 'close_price', 'open_price',
    'high_price', 'low_price', 'settlement_price', 'direction'
]
feature_cols = [col for col in train.columns if col not in cols_to_remove]

# Remove last row (no future return available) and drop missing values
train = train[:-1].dropna()
test  = test[:-1].dropna()

X_train = train[feature_cols].values
y_train = train['direction'].values
X_test  = test[feature_cols].values
y_test  = test['direction'].values

# Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'\nX_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'Training direction distribution — Up: {y_train.sum()}, Down: {len(y_train) - y_train.sum()}')
print(f'Testing direction distribution  — Up: {y_test.sum()}, Down: {len(y_test) - y_test.sum()}')

### 2.2 Traditional ML Models

In [ ]:
# ============================================================================
# Train 6 Traditional ML Models
# ============================================================================
results = []
trained_models = {}

model_configs = {
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
    ),
    'SVM (Linear)': SVC(kernel='linear', probability=True, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=42, verbose=-1
    ),
}

for name, model in model_configs.items():
    print(f'Training {name}...')
    model.fit(X_train_scaled, y_train)

    y_pred_train = model.predict(X_train_scaled)
    y_pred_test  = model.predict(X_test_scaled)
    y_prob_train = model.predict_proba(X_train_scaled)[:, 1]
    y_prob_test  = model.predict_proba(X_test_scaled)[:, 1]

    results.append({
        'Model': name,
        'Train_Accuracy': accuracy_score(y_train, y_pred_train),
        'Test_Accuracy':  accuracy_score(y_test, y_pred_test),
        'Test_Precision': precision_score(y_test, y_pred_test, zero_division=0),
        'Test_Recall':    recall_score(y_test, y_pred_test, zero_division=0),
        'Test_F1':        f1_score(y_test, y_pred_test, zero_division=0),
        'Train_AUC':      roc_auc_score(y_train, y_prob_train),
        'Test_AUC':       roc_auc_score(y_test, y_prob_test),
    })
    trained_models[name] = model
    print(f'  Test Accuracy: {results[-1]["Test_Accuracy"]:.4f}  |  AUC: {results[-1]["Test_AUC"]:.4f}')

print('\nAll traditional models trained.')

### 2.3 Deep Learning Models (Sequence-Based)

In [ ]:
# ============================================================================
# Create Sequence Data for Time-Series Models
# ============================================================================
def create_sequences(X, y, seq_length=20):
    """Create sliding-window sequences for LSTM/CNN models."""
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i + seq_length])
        y_seq.append(y[i + seq_length])
    return np.array(X_seq), np.array(y_seq)

SEQ_LENGTH = 20
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, SEQ_LENGTH)
X_test_seq,  y_test_seq  = create_sequences(X_test_scaled,  y_test,  SEQ_LENGTH)

print(f'Sequence training data shape: {X_train_seq.shape}')
print(f'Sequence testing data shape:  {X_test_seq.shape}')

In [ ]:
# ============================================================================
# Model 7: Simple LSTM
# ============================================================================
n_features = X_train_scaled.shape[1]

model_lstm = keras.Sequential([
    layers.LSTM(64, input_shape=(SEQ_LENGTH, n_features)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])
model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.fit(X_train_seq, y_train_seq, epochs=50, batch_size=32,
               validation_split=0.2, verbose=0)

y_prob_lstm = model_lstm.predict(X_test_seq, verbose=0).flatten()
y_pred_lstm = (y_prob_lstm > 0.5).astype(int)

results.append({
    'Model': 'Simple LSTM',
    'Train_Accuracy': accuracy_score(y_train_seq, (model_lstm.predict(X_train_seq, verbose=0) > 0.5).astype(int).flatten()),
    'Test_Accuracy':  accuracy_score(y_test_seq, y_pred_lstm),
    'Test_Precision': precision_score(y_test_seq, y_pred_lstm, zero_division=0),
    'Test_Recall':    recall_score(y_test_seq, y_pred_lstm, zero_division=0),
    'Test_F1':        f1_score(y_test_seq, y_pred_lstm, zero_division=0),
    'Train_AUC':      roc_auc_score(y_train_seq, model_lstm.predict(X_train_seq, verbose=0).flatten()),
    'Test_AUC':       roc_auc_score(y_test_seq, y_prob_lstm),
})
print(f'Simple LSTM — Test Acc: {results[-1]["Test_Accuracy"]:.4f}  |  AUC: {results[-1]["Test_AUC"]:.4f}')

In [ ]:
# ============================================================================
# Model 8: Bidirectional LSTM
# ============================================================================
model_bilstm = keras.Sequential([
    layers.Bidirectional(
        layers.LSTM(64, return_sequences=False),
        input_shape=(SEQ_LENGTH, n_features)
    ),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])
model_bilstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_bilstm.fit(X_train_seq, y_train_seq, epochs=50, batch_size=32,
                  validation_split=0.2, verbose=0)

y_prob_bilstm = model_bilstm.predict(X_test_seq, verbose=0).flatten()
y_pred_bilstm = (y_prob_bilstm > 0.5).astype(int)

results.append({
    'Model': 'Bidirectional LSTM',
    'Train_Accuracy': accuracy_score(y_train_seq, (model_bilstm.predict(X_train_seq, verbose=0) > 0.5).astype(int).flatten()),
    'Test_Accuracy':  accuracy_score(y_test_seq, y_pred_bilstm),
    'Test_Precision': precision_score(y_test_seq, y_pred_bilstm, zero_division=0),
    'Test_Recall':    recall_score(y_test_seq, y_pred_bilstm, zero_division=0),
    'Test_F1':        f1_score(y_test_seq, y_pred_bilstm, zero_division=0),
    'Train_AUC':      roc_auc_score(y_train_seq, model_bilstm.predict(X_train_seq, verbose=0).flatten()),
    'Test_AUC':       roc_auc_score(y_test_seq, y_prob_bilstm),
})
print(f'Bidirectional LSTM — Test Acc: {results[-1]["Test_Accuracy"]:.4f}  |  AUC: {results[-1]["Test_AUC"]:.4f}')

In [ ]:
# ============================================================================
# Model 9: LSTM + Attention
# ============================================================================
class AttentionLayer(layers.Layer):
    """Bahdanau-style additive attention over temporal steps."""
    def build(self, input_shape):
        self.W = self.add_weight('attention_weight',
                                 shape=(input_shape[-1], input_shape[-1]),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight('attention_bias',
                                 shape=(input_shape[-1],),
                                 initializer='zeros', trainable=True)
        super().build(input_shape)

    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        return tf.reduce_sum(x * a, axis=1)

inp = layers.Input(shape=(SEQ_LENGTH, n_features))
x   = layers.LSTM(64, return_sequences=True)(inp)
x   = layers.Dropout(0.3)(x)
x   = AttentionLayer()(x)
x   = layers.Dense(32, activation='relu')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation='sigmoid')(x)

model_attention = keras.Model(inputs=inp, outputs=out)
model_attention.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_attention.fit(X_train_seq, y_train_seq, epochs=50, batch_size=32,
                    validation_split=0.2, verbose=0)

y_prob_attn = model_attention.predict(X_test_seq, verbose=0).flatten()
y_pred_attn = (y_prob_attn > 0.5).astype(int)

results.append({
    'Model': 'LSTM + Attention',
    'Train_Accuracy': accuracy_score(y_train_seq, (model_attention.predict(X_train_seq, verbose=0) > 0.5).astype(int).flatten()),
    'Test_Accuracy':  accuracy_score(y_test_seq, y_pred_attn),
    'Test_Precision': precision_score(y_test_seq, y_pred_attn, zero_division=0),
    'Test_Recall':    recall_score(y_test_seq, y_pred_attn, zero_division=0),
    'Test_F1':        f1_score(y_test_seq, y_pred_attn, zero_division=0),
    'Train_AUC':      roc_auc_score(y_train_seq, model_attention.predict(X_train_seq, verbose=0).flatten()),
    'Test_AUC':       roc_auc_score(y_test_seq, y_prob_attn),
})
print(f'LSTM + Attention — Test Acc: {results[-1]["Test_Accuracy"]:.4f}  |  AUC: {results[-1]["Test_AUC"]:.4f}')

In [ ]:
# ============================================================================
# Model 10: 1D-CNN
# ============================================================================
model_cnn = keras.Sequential([
    layers.Conv1D(64, 3, activation='relu', input_shape=(SEQ_LENGTH, n_features)),
    layers.MaxPooling1D(2),
    layers.Conv1D(32, 3, activation='relu'),
    layers.MaxPooling1D(2),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])
model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_cnn.fit(X_train_seq, y_train_seq, epochs=50, batch_size=32,
              validation_split=0.2, verbose=0)

y_prob_cnn = model_cnn.predict(X_test_seq, verbose=0).flatten()
y_pred_cnn = (y_prob_cnn > 0.5).astype(int)

results.append({
    'Model': '1D-CNN',
    'Train_Accuracy': accuracy_score(y_train_seq, (model_cnn.predict(X_train_seq, verbose=0) > 0.5).astype(int).flatten()),
    'Test_Accuracy':  accuracy_score(y_test_seq, y_pred_cnn),
    'Test_Precision': precision_score(y_test_seq, y_pred_cnn, zero_division=0),
    'Test_Recall':    recall_score(y_test_seq, y_pred_cnn, zero_division=0),
    'Test_F1':        f1_score(y_test_seq, y_pred_cnn, zero_division=0),
    'Train_AUC':      roc_auc_score(y_train_seq, model_cnn.predict(X_train_seq, verbose=0).flatten()),
    'Test_AUC':       roc_auc_score(y_test_seq, y_prob_cnn),
})
print(f'1D-CNN — Test Acc: {results[-1]["Test_Accuracy"]:.4f}  |  AUC: {results[-1]["Test_AUC"]:.4f}')

### 2.4 Results Comparison

In [ ]:
# ============================================================================
# Direction Prediction — Results Summary
# ============================================================================
results_direction = pd.DataFrame(results)
results_direction = results_direction.sort_values('Test_Accuracy', ascending=False)
print(results_direction.to_string(index=False))

# Save to CSV
results_direction.to_csv(f'{OUTPUT_DIR}/direction_prediction_results.csv', index=False)

### 2.5 ROC Curve Comparison

In [ ]:
# ============================================================================
# ROC Curves — All Models
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 8))

# Traditional ML models
for name, model in trained_models.items():
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', linewidth=2)

# Deep learning models
dl_data = [
    ('Simple LSTM', y_prob_lstm),
    ('Bidirectional LSTM', y_prob_bilstm),
    ('LSTM + Attention', y_prob_attn),
    ('1D-CNN', y_prob_cnn),
]
for name, y_prob in dl_data:
    fpr, tpr, _ = roc_curve(y_test_seq, y_prob)
    auc_val = roc_auc_score(y_test_seq, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', linewidth=2, linestyle='--')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Direction Prediction (All Models)', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/roc_curves_direction.png', dpi=300, bbox_inches='tight')
plt.show()

### 2.6 SHAP Interpretability Analysis

In [ ]:
# ============================================================================
# SHAP Feature Importance (LightGBM)
# ============================================================================
lgbm_model = trained_models['LightGBM']
explainer   = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(X_test_scaled)

if isinstance(shap_values, list):
    shap_values = shap_values[1]

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_scaled, feature_names=feature_cols,
                  show=False, max_display=20)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/shap_summary_direction.png', dpi=300, bbox_inches='tight')
plt.show()

### 2.7 Trading Strategy Backtesting

In [ ]:
# ============================================================================
# Simple Direction-Based Trading Strategy (LightGBM)
# ============================================================================
test_prices = test['close_price'].values[:-1]
strategy_predictions = lgbm_model.predict(X_test_scaled)
actual_returns = np.diff(test_prices) / test_prices[:-1]

min_len = min(len(strategy_predictions), len(actual_returns))
strategy_predictions = strategy_predictions[:min_len]
actual_returns = actual_returns[:min_len]

# Long when predict up, short when predict down
strategy_returns = np.where(strategy_predictions == 1, actual_returns, -actual_returns)
buy_hold_returns  = actual_returns

cumulative_strategy = np.cumprod(1 + strategy_returns) - 1
cumulative_buyhold  = np.cumprod(1 + buy_hold_returns) - 1

# --- Visualization ---
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Cumulative returns
axes[0].plot(cumulative_strategy, label='Direction Prediction Strategy', linewidth=2)
axes[0].plot(cumulative_buyhold,  label='Buy & Hold', linewidth=2)
axes[0].set_ylabel('Cumulative Return')
axes[0].set_title('Cumulative Returns Comparison', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Drawdown
def calc_drawdown(returns):
    cum = np.cumprod(1 + returns)
    running_max = np.maximum.accumulate(cum)
    return (cum - running_max) / running_max * 100

axes[1].plot(calc_drawdown(strategy_returns), label='Strategy', linewidth=2)
axes[1].plot(calc_drawdown(buy_hold_returns), label='Buy & Hold', linewidth=2)
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_title('Drawdown Comparison', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Rolling Sharpe
def rolling_sharpe(returns, window=30):
    r = pd.Series(returns)
    return (r.rolling(window).mean() / r.rolling(window).std()) * np.sqrt(252)

axes[2].plot(rolling_sharpe(strategy_returns), label='Strategy', linewidth=2)
axes[2].plot(rolling_sharpe(buy_hold_returns), label='Buy & Hold', linewidth=2)
axes[2].set_ylabel('Sharpe Ratio')
axes[2].set_xlabel('Trading Days')
axes[2].set_title('Rolling Sharpe Ratio (30-Day Window)', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/trading_strategy_direction.png', dpi=300, bbox_inches='tight')
plt.show()

# Performance metrics
def strategy_metrics(returns, name):
    total = (np.prod(1 + returns) - 1) * 100
    annual = ((1 + total / 100) ** (252 / len(returns)) - 1) * 100
    vol = np.std(returns) * np.sqrt(252) * 100
    sharpe = (np.mean(returns) / np.std(returns)) * np.sqrt(252) if np.std(returns) > 0 else 0
    dd = np.min(calc_drawdown(returns))
    wr = (returns > 0).sum() / len(returns) * 100
    return {'Strategy': name, 'Total Return (%)': total, 'Annual Return (%)': annual,
            'Volatility (%)': vol, 'Sharpe Ratio': sharpe, 'Max Drawdown (%)': dd, 'Win Rate (%)': wr}

comparison = pd.DataFrame([
    strategy_metrics(strategy_returns, 'Direction Prediction'),
    strategy_metrics(buy_hold_returns, 'Buy & Hold'),
])
print(comparison.to_string(index=False))
comparison.to_csv(f'{OUTPUT_DIR}/strategy_comparison_direction.csv', index=False)

---
## 3. Task 2 — Volatility Prediction (Regression)

**Objective:** Predict the realized volatility of CSI 300 futures on the next trading day.

**Target variable:** Realized volatility (continuous, computed from intraday price range).

**Models evaluated:** Linear Regression, Ridge, Lasso, ElasticNet, Decision Tree, Random Forest, Gradient Boosting, XGBoost, LightGBM, CatBoost.

**Regime analysis:** We partition the test set into Low / Medium / High volatility regimes using quantile-based thresholds and evaluate each model's performance per regime.

### 3.1 Data Loading and Feature Preparation

In [ ]:
# ============================================================================
# Data Loading — Volatility Prediction
# ============================================================================
VOL_TRAIN_FILE = 'data/vol_train_2015_2023.csv'
VOL_TEST_FILE  = 'data/vol_test_2024_2025.csv'

df_vol_train = pd.read_csv(VOL_TRAIN_FILE)
df_vol_test  = pd.read_csv(VOL_TEST_FILE)

print(f'Volatility training data shape: {df_vol_train.shape}')
print(f'Volatility testing data shape:  {df_vol_test.shape}')

# Remove data-leakage features
LEAKAGE_FEATURES = [
    'std_deviation', 'volatility_percentile',
    'high_volatility_regime', 'low_volatility_regime', 'volatility_target'
]
EXCLUDE_COLS = ['date', 'target', 'volatility', 'high_volatility']
TARGET_COL   = 'volatility'

df_vol_train = df_vol_train.drop(columns=LEAKAGE_FEATURES, errors='ignore')
df_vol_test  = df_vol_test.drop(columns=LEAKAGE_FEATURES, errors='ignore')

vol_feature_cols = [c for c in df_vol_train.columns if c not in EXCLUDE_COLS + [TARGET_COL]]

X_vol_train = df_vol_train[vol_feature_cols].values
y_vol_train = df_vol_train[TARGET_COL].values
X_vol_test  = df_vol_test[vol_feature_cols].values
y_vol_test  = df_vol_test[TARGET_COL].values

# Standardize
vol_scaler = StandardScaler()
X_vol_train_s = vol_scaler.fit_transform(X_vol_train)
X_vol_test_s  = vol_scaler.transform(X_vol_test)

print(f'\nFeatures: {len(vol_feature_cols)}')
print(f'X_vol_train: {X_vol_train_s.shape}, X_vol_test: {X_vol_test_s.shape}')

### 3.2 Model Training and Evaluation

In [ ]:
# ============================================================================
# Train 10 Regression Models
# ============================================================================
RANDOM_STATE = 42

vol_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=10.0, random_state=RANDOM_STATE),
    'Lasso Regression': Lasso(alpha=1.0, random_state=RANDOM_STATE),
    'ElasticNet': ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, min_samples_split=20,
                                           min_samples_leaf=10, random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10,
                                           min_samples_split=20, min_samples_leaf=10,
                                           random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        min_samples_split=20, subsample=0.8, random_state=RANDOM_STATE),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        min_child_weight=10, subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=-1),
    'LightGBM': lgb.LGBMRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        num_leaves=31, min_child_samples=20, subsample=0.8,
        colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1, n_jobs=-1),
    'CatBoost': CatBoostRegressor(
        iterations=200, learning_rate=0.05, depth=5,
        min_data_in_leaf=20, random_state=RANDOM_STATE, verbose=0),
}

vol_results = []
vol_trained = {}

for name, model in vol_models.items():
    print(f'Training {name}...')
    model.fit(X_vol_train_s, y_vol_train)

    y_pred_tr = model.predict(X_vol_train_s)
    y_pred_te = model.predict(X_vol_test_s)

    r2_tr  = r2_score(y_vol_train, y_pred_tr)
    r2_te  = r2_score(y_vol_test,  y_pred_te)
    rmse   = np.sqrt(mean_squared_error(y_vol_test, y_pred_te))
    mae    = mean_absolute_error(y_vol_test, y_pred_te)

    # Direction accuracy
    dir_true = np.sign(np.diff(y_vol_test))
    dir_pred = np.sign(np.diff(y_pred_te))
    dir_acc  = np.mean(dir_true == dir_pred)

    vol_results.append({
        'Model': name, 'Train_R2': r2_tr, 'Test_R2': r2_te,
        'Test_RMSE': rmse, 'Test_MAE': mae,
        'Direction_Accuracy': dir_acc,
        'Overfitting_Gap': r2_tr - r2_te,
    })
    vol_trained[name] = (model, y_pred_te)
    print(f'  Train R²: {r2_tr:.4f}  |  Test R²: {r2_te:.4f}  |  RMSE: {rmse:.2f}')

vol_results_df = pd.DataFrame(vol_results).sort_values('Test_R2', ascending=False)
print('\n' + vol_results_df.to_string(index=False))
vol_results_df.to_csv(f'{OUTPUT_DIR}/volatility_prediction_results.csv', index=False)

### 3.3 Regime-Aware Performance Analysis

In [ ]:
# ============================================================================
# Regime Analysis — Evaluate Best Model Across Market Regimes
# ============================================================================
best_vol_model_name = vol_results_df.iloc[0]['Model']
best_vol_pred = vol_trained[best_vol_model_name][1]

# Identify regimes via volatility quantiles
REGIME_NAMES = ['Low Volatility', 'Medium Volatility', 'High Volatility']
quantiles = np.quantile(y_vol_test, [0, 1/3, 2/3, 1])
regimes = np.zeros(len(y_vol_test), dtype=int)
for i in range(3):
    if i < 2:
        regimes[(y_vol_test >= quantiles[i]) & (y_vol_test < quantiles[i+1])] = i
    else:
        regimes[y_vol_test >= quantiles[i]] = i

regime_rows = []
for i, rname in enumerate(REGIME_NAMES):
    mask = regimes == i
    if mask.sum() == 0:
        continue
    r2_r  = r2_score(y_vol_test[mask], best_vol_pred[mask])
    rmse_r = np.sqrt(mean_squared_error(y_vol_test[mask], best_vol_pred[mask]))
    mae_r  = mean_absolute_error(y_vol_test[mask], best_vol_pred[mask])
    regime_rows.append({'Regime': rname, 'N': mask.sum(), 'R2': r2_r, 'RMSE': rmse_r, 'MAE': mae_r})

regime_df = pd.DataFrame(regime_rows)
print(f'\n{best_vol_model_name} — Performance by Market Regime:')
print(regime_df.to_string(index=False))
regime_df.to_csv(f'{OUTPUT_DIR}/regime_analysis_volatility.csv', index=False)

### 3.4 Visualizations

In [ ]:
# ============================================================================
# Volatility Prediction Visualizations
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. R² comparison
ax = axes[0, 0]
colors = plt.cm.RdYlGn(vol_results_df['Test_R2'] / vol_results_df['Test_R2'].max())
ax.barh(range(len(vol_results_df)), vol_results_df['Test_R2'], color=colors, edgecolor='black')
ax.set_yticks(range(len(vol_results_df)))
ax.set_yticklabels(vol_results_df['Model'], fontsize=9)
ax.set_xlabel('R² Score')
ax.set_title('Test R² Comparison', fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# 2. RMSE comparison
ax = axes[0, 1]
colors_r = plt.cm.RdYlGn_r(vol_results_df['Test_RMSE'] / vol_results_df['Test_RMSE'].max())
ax.barh(range(len(vol_results_df)), vol_results_df['Test_RMSE'], color=colors_r, edgecolor='black')
ax.set_yticks(range(len(vol_results_df)))
ax.set_yticklabels(vol_results_df['Model'], fontsize=9)
ax.set_xlabel('RMSE')
ax.set_title('Test RMSE Comparison', fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# 3. Time-series: Actual vs Predicted (best model)
ax = axes[1, 0]
ax.plot(y_vol_test, label='Actual', alpha=0.7, linewidth=1.5)
ax.plot(best_vol_pred, label='Predicted', alpha=0.7, linewidth=1.5)
ax.set_xlabel('Time Index')
ax.set_ylabel('Volatility')
ax.set_title(f'{best_vol_model_name} — Actual vs Predicted', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Overfitting gap
ax = axes[1, 1]
gap = vol_results_df.sort_values('Overfitting_Gap', ascending=True)
colors_g = ['#e74c3c' if v > 0.15 else '#2ecc71' for v in gap['Overfitting_Gap']]
ax.barh(range(len(gap)), gap['Overfitting_Gap'], color=colors_g, edgecolor='black')
ax.set_yticks(range(len(gap)))
ax.set_yticklabels(gap['Model'], fontsize=9)
ax.set_xlabel('Overfitting Gap (Train R² − Test R²)')
ax.set_title('Overfitting Analysis', fontweight='bold')
ax.axvline(x=0.15, color='red', linestyle='--', alpha=0.5, label='Threshold')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/volatility_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 4. Task 3 — Sentiment Breakthrough Prediction (Classification + Trading)

**Objective:** Predict whether market sentiment (a composite indicator derived from RSI, KDJ, CCI, and Williams %R) will experience a significant upward breakthrough on the next trading day.

**Target variable:** `target = 1` if `sentiment_score(t+1) − sentiment_score(t) > threshold` (we use threshold = 10), else `0`.

**Trading logic:** When the model predicts an imminent sentiment breakthrough, take a long position. Sentiment surges are empirically associated with positive price returns.

**Models evaluated:** Logistic Regression, Random Forest, XGBoost, LightGBM, CatBoost.

**Strategies evaluated:** Simple Signal, High-Confidence (threshold 0.6/0.7), Long-Short, Low-Volatility Filter, Dynamic Position Sizing.

### 4.1 Data Preparation

In [ ]:
# ============================================================================
# Sentiment Score Calculation and Target Generation
# ============================================================================
SENTIMENT_DATA = 'data/sentiment_training_data.csv'
SENTIMENT_THRESHOLD = 10

df_sent = pd.read_csv(SENTIMENT_DATA)
df_sent['date'] = pd.to_datetime(df_sent['date'])
df_sent = df_sent.sort_values('date').reset_index(drop=True)

print(f'Raw data shape: {df_sent.shape}')
print(f'Date range: {df_sent["date"].min()} to {df_sent["date"].max()}')

# Compute composite sentiment score (if not pre-computed)
if 'sentiment_score' not in df_sent.columns:
    df_sent['sentiment_score'] = (
        0.35 * df_sent['RSI1'] +
        0.35 * df_sent['K'] +
        0.20 * ((df_sent['CCI'] + 200) / 4) +
        0.10 * (100 - df_sent['W_R'])
    )

# Target: next-day sentiment breakthrough
df_sent['sentiment_change_next'] = df_sent['sentiment_score'].diff().shift(-1)
df_sent['target'] = (df_sent['sentiment_change_next'] > SENTIMENT_THRESHOLD).astype(int)

# Next-day return for backtesting
df_sent['return_next'] = df_sent['close_price'].pct_change().shift(-1)

# Next-day volatility for low-vol filter strategy
df_sent['volatility_target'] = (
    (df_sent['high_price'] - df_sent['low_price']) / df_sent['close_price']
).shift(-1)

df_sent = df_sent[:-1].copy()  # Remove last row (no future data)

pos_samples = df_sent[df_sent['target'] == 1]
print(f'\nTarget distribution:')
print(f'  Positive (breakthrough): {df_sent["target"].sum()} ({df_sent["target"].mean()*100:.1f}%)')
print(f'  Negative:               {(df_sent["target"]==0).sum()} ({(df_sent["target"]==0).mean()*100:.1f}%)')
if len(pos_samples) > 0:
    print(f'\nWhen target=1 (breakthrough predicted):')
    print(f'  Next-day win rate:    {(pos_samples["return_next"] > 0).mean()*100:.2f}%')
    print(f'  Next-day avg return:  {pos_samples["return_next"].mean()*100:.4f}%')

### 4.2 Time-Series Train / Validation / Test Split

In [ ]:
# ============================================================================
# Strict Temporal Split (70% / 15% / 15%)
# ============================================================================
sent_exclude = [
    'date', 'target', 'sentiment_change_next', 'return_next',
    'direction_target', 'volatility_target', 'high_low_range',
    'pct_change', 'close_price', 'high_price', 'low_price',
    'regime_volatility', 'regime_trend', 'regime_composite'
]
sent_features = [c for c in df_sent.columns if c not in sent_exclude]

n = len(df_sent)
tr_end  = int(n * 0.70)
val_end = int(n * 0.85)

df_s_train = df_sent.iloc[:tr_end]
df_s_val   = df_sent.iloc[tr_end:val_end]
df_s_test  = df_sent.iloc[val_end:]

X_s_train, y_s_train = df_s_train[sent_features], df_s_train['target']
X_s_val,   y_s_val   = df_s_val[sent_features],   df_s_val['target']
X_s_test,  y_s_test  = df_s_test[sent_features],  df_s_test['target']

print(f'Train: {len(X_s_train)} samples  ({df_s_train["date"].min()} — {df_s_train["date"].max()})  Positive: {y_s_train.mean()*100:.1f}%')
print(f'Val:   {len(X_s_val)} samples  ({df_s_val["date"].min()} — {df_s_val["date"].max()})  Positive: {y_s_val.mean()*100:.1f}%')
print(f'Test:  {len(X_s_test)} samples  ({df_s_test["date"].min()} — {df_s_test["date"].max()})  Positive: {y_s_test.mean()*100:.1f}%')

### 4.3 Model Training (5 Classifiers)

In [ ]:
# ============================================================================
# Train 5 Classifiers for Sentiment Breakthrough
# ============================================================================
scale_pw = (len(y_s_train) - y_s_train.sum()) / y_s_train.sum()

sent_models = {
    'Logistic Regression': LogisticRegression(
        solver='lbfgs', penalty='l2', C=1.0, class_weight='balanced',
        max_iter=500, random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_split=10,
        min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=300, learning_rate=0.03, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pw,
        use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=8,
        num_leaves=31, min_child_samples=20, subsample=0.8,
        colsample_bytree=0.8, class_weight='balanced',
        random_state=42, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=6,
        scale_pos_weight=scale_pw, random_seed=42, verbose=0),
}

# Standardize
sent_scaler = StandardScaler()
Xs_train_s = sent_scaler.fit_transform(X_s_train)
Xs_val_s   = sent_scaler.transform(X_s_val)
Xs_test_s  = sent_scaler.transform(X_s_test)

sent_results = []
sent_predictions = {}

for name, model in sent_models.items():
    print(f'Training {name}...')

    # Models with eval_set support
    if name == 'LightGBM':
        model.fit(Xs_train_s, y_s_train, eval_set=[(Xs_val_s, y_s_val)],
                  eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
    elif name == 'XGBoost':
        model.fit(Xs_train_s, y_s_train, eval_set=[(Xs_val_s, y_s_val)], verbose=False)
    elif name == 'CatBoost':
        model.fit(Xs_train_s, y_s_train, eval_set=(Xs_val_s, y_s_val),
                  early_stopping_rounds=50)
    else:
        model.fit(Xs_train_s, y_s_train)

    y_prob_s = model.predict_proba(Xs_test_s)[:, 1]
    y_pred_s = (y_prob_s >= 0.5).astype(int)

    auc_s  = roc_auc_score(y_s_test, y_prob_s)
    acc_s  = accuracy_score(y_s_test, y_pred_s)
    prec_s = precision_score(y_s_test, y_pred_s, zero_division=0)
    rec_s  = recall_score(y_s_test, y_pred_s, zero_division=0)
    f1_s   = f1_score(y_s_test, y_pred_s, zero_division=0)

    sent_results.append({'Model': name, 'AUC': auc_s, 'Accuracy': acc_s,
                         'Precision': prec_s, 'Recall': rec_s, 'F1_Score': f1_s})
    sent_predictions[name] = y_prob_s
    print(f'  AUC: {auc_s:.4f}  |  F1: {f1_s:.4f}')

sent_results_df = pd.DataFrame(sent_results).sort_values('AUC', ascending=False)
print('\n' + sent_results_df.to_string(index=False))
sent_results_df.to_csv(f'{OUTPUT_DIR}/sentiment_classification_results.csv', index=False)

### 4.4 Trading Strategy Backtesting

In [ ]:
# ============================================================================
# Trading Backtester — 5 Strategy Variants
# ============================================================================
class TradingBacktester:
    """Backtest trading strategies based on sentiment breakthrough predictions."""

    def __init__(self, df_test, y_pred_proba):
        self.df = df_test.copy()
        self.df['pred_proba'] = y_pred_proba
        self.df['pred_label'] = (y_pred_proba >= 0.5).astype(int)
        self.benchmark = self.df['return_next'].fillna(0)

    def _returns(self, condition):
        return pd.Series(
            np.where(condition, self.df['return_next'], 0),
            index=self.df.index
        )

    def strategy_simple(self):
        return self._returns(self.df['pred_label'] == 1)

    def strategy_high_confidence(self, threshold=0.7):
        return self._returns(self.df['pred_proba'] > threshold)

    def strategy_long_short(self, long_th=0.6, short_th=0.4):
        r = np.where(
            self.df['pred_proba'] > long_th, self.df['return_next'],
            np.where(self.df['pred_proba'] < short_th, -self.df['return_next'], 0)
        )
        return pd.Series(r, index=self.df.index)

    def strategy_low_vol(self, vol_th=0.015):
        low_vol = self.df['volatility_target'] < vol_th
        return self._returns((self.df['pred_label'] == 1) & low_vol)

    def strategy_dynamic_position(self):
        return pd.Series(
            self.df['pred_proba'] * self.df['return_next'],
            index=self.df.index
        )

    def calc_metrics(self, rets, name):
        total = (1 + rets).prod() - 1
        n = len(rets)
        ann_ret = (1 + total) ** (252 / n) - 1
        ann_vol = rets.std() * np.sqrt(252)
        cum = (1 + rets).cumprod()
        dd = ((cum - cum.expanding().max()) / cum.expanding().max()).min()
        sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
        calmar = ann_ret / abs(dd) if dd < 0 else np.inf
        trades = (rets != 0).sum()
        wr = (rets > 0).sum() / trades if trades > 0 else 0
        return {
            'Strategy': name, 'Annual_Return': ann_ret, 'Sharpe': sharpe,
            'Max_Drawdown': dd, 'Calmar': calmar, 'Win_Rate': wr, 'N_Trades': trades
        }

    def run_all(self):
        strategies = {
            'Benchmark (Buy & Hold)': self.benchmark,
            'Simple Signal':          self.strategy_simple(),
            'High Confidence (0.7)':  self.strategy_high_confidence(0.7),
            'Long-Short':             self.strategy_long_short(0.6, 0.4),
            'Low Volatility Filter':  self.strategy_low_vol(0.015),
            'Dynamic Position':       self.strategy_dynamic_position(),
        }
        rows = [self.calc_metrics(r, n) for n, r in strategies.items()]
        return pd.DataFrame(rows), strategies


# Run backtest with the best model (Random Forest by AUC)
best_sent_model = sent_results_df.iloc[0]['Model']
print(f'Backtesting with best model: {best_sent_model}')

bt = TradingBacktester(df_s_test, sent_predictions[best_sent_model])
bt_results, bt_strategies = bt.run_all()

print('\nTrading Strategy Performance:')
print(bt_results.to_string(index=False))
bt_results.to_csv(f'{OUTPUT_DIR}/sentiment_strategy_results.csv', index=False)

### 4.5 Strategy Visualization

In [ ]:
# ============================================================================
# Backtest Visualization — Sentiment Strategies
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Cumulative returns
ax = axes[0, 0]
for name, rets in bt_strategies.items():
    cum = (1 + rets).cumprod()
    style = {'linewidth': 2, 'linestyle': '--', 'alpha': 0.6, 'color': 'gray'} \
        if 'Benchmark' in name else {'linewidth': 1.5, 'alpha': 0.8}
    ax.plot(cum.values, label=name, **style)
ax.set_title('Cumulative Returns', fontweight='bold')
ax.set_ylabel('Growth of $1')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 2. Drawdown
ax = axes[0, 1]
for name, rets in bt_strategies.items():
    if 'Benchmark' in name:
        continue
    cum = (1 + rets).cumprod()
    dd = (cum - cum.expanding().max()) / cum.expanding().max()
    ax.plot(dd.values, label=name, linewidth=1.5, alpha=0.8)
ax.set_title('Drawdown Curves', fontweight='bold')
ax.set_ylabel('Drawdown')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 3. Sharpe comparison
ax = axes[1, 0]
strat_only = bt_results[~bt_results['Strategy'].str.contains('Benchmark')]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
ax.barh(range(len(strat_only)), strat_only['Sharpe'], color=colors[:len(strat_only)], alpha=0.8)
ax.set_yticks(range(len(strat_only)))
ax.set_yticklabels(strat_only['Strategy'])
ax.set_xlabel('Sharpe Ratio')
ax.set_title('Sharpe Ratio Comparison', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# 4. Max drawdown comparison
ax = axes[1, 1]
ax.barh(range(len(strat_only)), strat_only['Max_Drawdown'] * 100,
        color=colors[:len(strat_only)], alpha=0.8)
ax.set_yticks(range(len(strat_only)))
ax.set_yticklabels(strat_only['Strategy'])
ax.set_xlabel('Max Drawdown (%)')
ax.set_title('Maximum Drawdown Comparison', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.suptitle(f'Trading Backtest — Sentiment Breakthrough ({best_sent_model})',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sentiment_backtest_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 5. Summary and Conclusions

### Task 1 — Price Direction Prediction

| Model | Train Acc | Test Acc | Precision | Recall | F1 | Train AUC | Test AUC |
|-------|-----------|----------|-----------|--------|-----|-----------|----------|
| Decision Tree | 79.30% | 52.11% | 0.516 | 0.497 | 0.506 | 0.897 | 0.513 |
| Simple LSTM | 72.00% | 51.96% | 0.722 | 0.068 | 0.124 | 0.807 | 0.490 |
| Gradient Boosting | 96.85% | 51.61% | 0.515 | 0.352 | 0.418 | 0.996 | 0.507 |
| SVM (Linear) | 57.13% | 51.36% | 0.541 | 0.101 | 0.169 | 0.608 | 0.477 |
| SVM (RBF) | 66.54% | 51.12% | 0.521 | 0.126 | 0.202 | 0.737 | 0.497 |
| Logistic Regression | 57.68% | 51.12% | 0.525 | 0.106 | 0.176 | 0.609 | 0.491 |
| 1D-CNN | 90.04% | 50.91% | 0.515 | 0.354 | 0.420 | 0.929 | 0.479 |
| LightGBM | 92.50% | 50.37% | 0.494 | 0.211 | 0.296 | 0.978 | 0.498 |
| LSTM + Attention | 67.62% | 49.87% | 0.000 | 0.000 | 0.000 | 0.733 | 0.559 |
| Bidirectional LSTM | 78.87% | 49.61% | 0.491 | 0.135 | 0.212 | 0.875 | 0.516 |

**Finding:** All models achieve near-random test accuracy (~50%), confirming the well-documented difficulty of short-term price direction prediction in efficient futures markets. The large train-test gap (especially for Gradient Boosting and LightGBM) indicates substantial overfitting to training-period patterns.

### Task 2 — Volatility Prediction

| Model | Train R² | Test R² | RMSE | MAE | Direction Acc | Overfit Gap |
|-------|----------|---------|------|-----|---------------|-------------|
| Linear Regression | 0.892 | 0.900 | 19.16 | 11.35 | 65.1% | −0.008 |
| Ridge Regression | 0.891 | 0.894 | 19.73 | 11.44 | 66.0% | −0.004 |
| Lasso Regression | 0.873 | 0.864 | 22.35 | 13.42 | 66.0% | 0.009 |
| LightGBM | 0.991 | 0.863 | 22.42 | 11.59 | 68.2% | 0.127 |
| XGBoost | 0.993 | 0.857 | 22.93 | 11.74 | 72.2% | 0.136 |
| Gradient Boosting | 0.994 | 0.857 | 22.94 | 11.93 | 68.6% | 0.137 |
| Random Forest | 0.966 | 0.802 | 27.02 | 14.09 | 66.5% | 0.165 |
| CatBoost | 0.969 | 0.768 | 29.20 | 12.91 | 67.9% | 0.201 |
| Decision Tree | 0.945 | 0.691 | 33.73 | 18.33 | 27.6% | 0.254 |
| ElasticNet | 0.802 | 0.682 | 34.24 | 18.90 | 63.4% | 0.121 |

**Finding:** Linear/Ridge regression achieve the best out-of-sample R² (~0.90) with minimal overfitting, outperforming complex ensemble methods that overfit training data. This result aligns with volatility's autoregressive nature — simple models capture the dominant signal effectively.

### Task 3 — Sentiment Breakthrough Prediction

| Model | AUC | Accuracy | Precision | Recall | F1 |
|-------|-----|----------|-----------|--------|----|
| Random Forest | 0.7189 | 88.59% | 0.273 | 0.167 | 0.207 |
| XGBoost | 0.6770 | 85.37% | 0.185 | 0.222 | 0.202 |
| Logistic Regression | 0.6356 | 85.37% | 0.212 | 0.204 | 0.208 |
| CatBoost | 0.6331 | 85.11% | 0.200 | 0.019 | 0.034 |
| LightGBM | 0.5853 | 85.11% | 0.147 | 0.019 | 0.036 |

**Trading Strategy (Random Forest, threshold = 0.6):**
- Annual Return: 2.71%
- Sharpe Ratio: 0.90 (vs. Benchmark 0.93)
- Maximum Drawdown: **−1.30%** (vs. Benchmark −18.98%)
- Hit Rate: 78.3%
- Trades: 27 over ~2 years

**Finding:** While classification metrics are modest due to severe class imbalance (~14% positive), the model demonstrates strong practical value as a risk management tool. The 93% reduction in maximum drawdown, combined with a competitive Sharpe ratio and 78.3% hit rate, validates sentiment breakthrough prediction as an effective signal for selective market entry.

---

*This notebook was produced as part of a machine learning group project. Data files are not included in this repository due to licensing restrictions. To reproduce, place your CSI 300 futures data in the `data/` directory with the expected column names.*